## Enterprise Business Intelligence Data Warehouse
#### Author: Sachin Kumar B
#### Date: 27/07/2026

In [1]:
# Step 1: Imports
import sqlite3
import pandas as pd

In [2]:
# Step 2: Initialize In-Memory SQL Data Warehouse
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
print("Connected to in-memory SQLite Data Warehouse instance.")

Connected to in-memory SQLite Data Warehouse instance.


In [3]:
# Step 3: DDL Schema Construction
cursor.executescript('''
CREATE TABLE dim_date (
    date_key INT PRIMARY KEY,
    full_date DATE NOT NULL,
    year INT NOT NULL,
    quarter INT NOT NULL,
    month INT NOT NULL
);

CREATE TABLE dim_customer (
    customer_key INT PRIMARY KEY,
    customer_id VARCHAR(20) NOT NULL,
    segment VARCHAR(50) NOT NULL,
    country VARCHAR(50) NOT NULL
);

CREATE TABLE fact_transactions (
    transaction_id VARCHAR(50) PRIMARY KEY,
    date_key INT REFERENCES dim_date(date_key),
    customer_key INT REFERENCES dim_customer(customer_key),
    amount NUMERIC(12,2) NOT NULL,
    processing_fee NUMERIC(8,2) NOT NULL
);
''')
conn.commit()
print("Star Schema tables created.")

Star Schema tables created.


In [4]:
# Step 4: Populate Dimension and Fact Seed Data
cursor.executescript('''
INSERT INTO dim_date VALUES 
(20260101, '2026-01-01', 2026, 1, 1),
(20260102, '2026-01-02', 2026, 1, 1);

INSERT INTO dim_customer VALUES 
(1, 'CUST-001', 'Enterprise', 'USA'),
(2, 'CUST-002', 'SMB', 'Canada'),
(3, 'CUST-003', 'Enterprise', 'UK');

INSERT INTO fact_transactions VALUES 
('TXN-1001', 20260101, 1, 15000.00, 450.00),
('TXN-1002', 20260101, 2, 2300.00, 69.00),
('TXN-1003', 20260102, 3, 42000.00, 1260.00),
('TXN-1004', 20260102, 1, 18500.00, 555.00);
''')
conn.commit()
print("Data seeded successfully.")

Data seeded successfully.


In [5]:
# Step 5: Create Optimized View & Query Analytics
cursor.execute('''
CREATE VIEW view_executive_monthly_revenue AS
SELECT 
    d.year,
    d.month,
    c.segment,
    COUNT(f.transaction_id) AS total_transactions,
    SUM(f.amount) AS gross_revenue,
    AVG(f.amount) AS avg_order_value
FROM fact_transactions f
JOIN dim_date d ON f.date_key = d.date_key
JOIN dim_customer c ON f.customer_key = c.customer_key
GROUP BY d.year, d.month, c.segment;
''')
conn.commit()

df_report = pd.read_sql_query("SELECT * FROM view_executive_monthly_revenue;", conn)
print("\nExecutive Summary Report:")
print(df_report.to_string(index=False))

conn.close()


Executive Summary Report:
 year  month    segment  total_transactions  gross_revenue  avg_order_value
 2026      1 Enterprise                   3          75500     25166.666667
 2026      1        SMB                   1           2300      2300.000000
